# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR<sup>2</sup> mass spectrometry analysis dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display key metadata fields
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name if hasattr(metadata, 'name') else ''}")
print(f"Dataset Description: {metadata.description if hasattr(metadata, 'description') else ''}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else ''}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List all available Record Sets and their fields/columns by @id
record_sets = dataset.record_sets
print("Available Record Sets (by @id):\n")
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields (by @id):")
    for field in rs.fields:
        colinfo = field.column.id if hasattr(field, 'column') and hasattr(field.column, 'id') else ''
        print(f"    @id: {field.id}, name: {field.name if hasattr(field, 'name') else ''}, column @id: {colinfo}")
    print()

# List the first 2 sample records from each record set using their @id
for rs in record_sets:
    print(f"\nSample records from record set @id={rs.id}:")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(json.dumps(rec, indent=2))
        if i >= 1:
            break

## 3. Data Extraction
Load data from all main record sets into pandas DataFrames for analysis. Use record set and field `@id`s as identified above.

In [ ]:
# Prepare DataFrames for each record set
# List all Record Set @ids
all_record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rsid in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for record set @id={rsid} (shape: {df.shape})")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading record set @id={rsid}: {e}\n")

# For demonstration, select the first record set for further exploration
if all_record_set_ids:
    selected_rs_id = all_record_set_ids[0]
    df = dataframes[selected_rs_id]
    print(f"\nPreview of DataFrame for record set @id={selected_rs_id}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by their `@id`.

In [ ]:
# Example: Filter, normalize, and group by a numeric field
# Automatically detect numeric fields in the selected record set

numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_fields:
    print("No numeric fields detected in the DataFrame.")
else:
    print(f"Numeric fields detected: {numeric_fields}")
    numeric_field = numeric_fields[0]  # Use the first numeric field for example

    # Set a simple threshold (e.g., median value)
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold]

    print(f"\nFiltered records with {numeric_field} > {threshold} (using field @id):")
    display(filtered_df.head())

    # Normalize the field
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normalized_col]].head())

    # Attempt grouping by another field, e.g., a possible 'category' or 'description' field
    possible_group_fields = [col for col in df.columns if (col != numeric_field and df[col].nunique() < df.shape[0] / 2)]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"{numeric_field}_mean")
        print(f"\nGrouped data by {group_field} (using field @id):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All columns are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric fields available for visualization.")
else:
    # Histogram of the selected numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (field @id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group field if available
    if possible_group_fields:
        group_field = possible_group_fields[0]
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (all by @id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR<sup>2</sup> dataset using `mlcroissant`, reviewed its structure, examined record sets by `@id`, performed basic data wrangling, and visualized distributions using referenced fields and columns. This workflow can be extended for domain-specific analyses leveraging the dataset's rich protein quantification and annotation context.